In [1]:
import sys
!{sys.executable} -m pip install -U "transformers>=4.42" "peft>=0.11.1" \
  "datasets>=2.19" "accelerate>=0.33" "bitsandbytes>=0.43" \
  tokenizers sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 123.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 181.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 153.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 kB 154.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 186.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA A100-SXM4-80GB


In [4]:
import sys
!{sys.executable} -m pip install -U huggingface_hub


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
from huggingface_hub import login, whoami
login()                 # 
print(whoami())         # verifica

{'type': 'user', 'id': '67fc03d5831bee28b57ccba1', 'name': 'benniAle', 'fullname': 'Alessandro Beninati', 'isPro': False, 'avatarUrl': '/avatars/9b21612dc301a1ce6a8892432dab55a6.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'anotherLove', 'role': 'fineGrained', 'createdAt': '2025-08-19T06:53:25.938Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '67fc03d5831bee28b57ccba1', 'type': 'user', 'name': 'benniAle'}, 'permissions': ['repo.content.read']}]}}}}


In [7]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"  # oppure "Qwen/Qwen2.5-7B-Instruct"
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tok.pad_token = tok.eos_token
print("Vocab ok")

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Vocab ok


In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
model.gradient_checkpointing_enable(); model.config.use_cache = False
print("Model ok")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model ok


In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
targets = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]  # Mistral
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=targets)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("Done")

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
Done


In [10]:
from datasets import load_dataset
import json
ds = load_dataset("json", data_files="/workspace/train.jsonl")["train"].train_test_split(0.05, seed=42)
tr, ev = ds["train"], ds["test"]

def fmt(ex):
    msgs=[{"role":"system","content":"Rispondi SOLO con JSON valido (nessun testo extra)."},
          {"role":"user","content":ex["input"]},
          {"role":"assistant","content":json.dumps(ex["output"], ensure_ascii=False)}]
    return tok.apply_chat_template(msgs, tokenize=False)

train_txt = tr.map(lambda e: {"text": fmt(e)}, remove_columns=tr.column_names)
val_txt   = ev.map(lambda e: {"text": fmt(e)}, remove_columns=ev.column_names)
print(len(train_txt), len(val_txt)); print(train_txt[0]["text"][:200])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/127004 [00:00<?, ? examples/s]

Map:   0%|          | 0/6685 [00:00<?, ? examples/s]

127004 6685
<s>[INST] <ARTICLE>
These stocks share at least two common denominators: a biopharmaceutical focus and great long-term prospects. In a sense, every investor is a growth investor. If you focus on valua


In [11]:
def tok_map(b): return tok(b["text"], truncation=True, max_length=2048)
train_tok = train_txt.map(tok_map, batched=True, remove_columns=["text"])
val_tok   = val_txt.map(tok_map,   batched=True, remove_columns=["text"])
print(train_tok[0].keys())

Map:   0%|          | 0/127004 [00:00<?, ? examples/s]

Map:   0%|          | 0/6685 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])


In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
dc = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

args = TrainingArguments(
    output_dir="out-mistral-json",
    bf16=True, per_device_train_batch_size=8, gradient_accumulation_steps=16,
    learning_rate=2e-4, max_steps=50,  # <-- debug corto
    logging_steps=10, save_steps=50, warmup_ratio=0.03, lr_scheduler_type="cosine",
    do_eval=True
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=dc)
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.496000
20,1.354200
30,1.338400
40,1.323400
50,1.332000


TrainOutput(global_step=50, training_loss=1.3687937927246094, metrics={'train_runtime': 5511.743, 'train_samples_per_second': 1.161, 'train_steps_per_second': 0.009, 'total_flos': 5.3705830138812826e+17, 'train_loss': 1.3687937927246094, 'epoch': 0.05039052658100277})

In [ ]:
trainer.save_model("out-mistral-json"); tok.save_pretrained("out-mistral-json")

article = tr[0]["input"]
prompt = tok.apply_chat_template(
  [{"role":"system","content":"Rispondi SOLO con JSON valido."},
   {"role":"user","content":article}], tokenize=False, add_generation_prompt=True)
out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                     max_new_tokens=180, temperature=0)
print(tok.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip())

In [11]:
print("ciao")

ciao


In [15]:
import os, glob
print("cwd:", os.getcwd())
print("files:", glob.glob("out-mistral-json/*"))


cwd: /workspace
files: ['out-mistral-json/training_args.bin', 'out-mistral-json/tokenizer.json', 'out-mistral-json/tokenizer.model', 'out-mistral-json/special_tokens_map.json', 'out-mistral-json/tokenizer_config.json', 'out-mistral-json/chat_template.jinja', 'out-mistral-json/adapter_config.json', 'out-mistral-json/adapter_model.safetensors', 'out-mistral-json/README.md', 'out-mistral-json/checkpoint-50']


In [16]:
import glob
glob.glob("out-mistral-json/checkpoint-50/*")


['out-mistral-json/checkpoint-50/trainer_state.json',
 'out-mistral-json/checkpoint-50/rng_state.pth',
 'out-mistral-json/checkpoint-50/scheduler.pt',
 'out-mistral-json/checkpoint-50/optimizer.pt',
 'out-mistral-json/checkpoint-50/training_args.bin',
 'out-mistral-json/checkpoint-50/tokenizer.json',
 'out-mistral-json/checkpoint-50/tokenizer.model',
 'out-mistral-json/checkpoint-50/special_tokens_map.json',
 'out-mistral-json/checkpoint-50/tokenizer_config.json',
 'out-mistral-json/checkpoint-50/chat_template.jinja',
 'out-mistral-json/checkpoint-50/adapter_config.json',
 'out-mistral-json/checkpoint-50/adapter_model.safetensors',
 'out-mistral-json/checkpoint-50/README.md']

In [17]:
trainer.save_model("out-mistral-json/final")
tok.save_pretrained("out-mistral-json/final")

('out-mistral-json/final/tokenizer_config.json',
 'out-mistral-json/final/special_tokens_map.json',
 'out-mistral-json/final/chat_template.jinja',
 'out-mistral-json/final/tokenizer.model',
 'out-mistral-json/final/added_tokens.json',
 'out-mistral-json/final/tokenizer.json')

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, json

base_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapters = "out-mistral-json/final"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(adapters); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, adapters).eval()

def infer(article:str):
    msgs=[{"role":"system","content":"Rispondi SOLO con JSON valido."},
          {"role":"user","content": f"<ARTICLE>{article}</ARTICLE>"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    with torch.inference_mode():
        out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                             max_new_tokens=200, temperature=0,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    return txt, (json.loads(txt) if txt.startswith("{") else None)

# Esempio:
jtxt, jobj = infer("American business leaders with “significant” investments in both the US and China – such as Elon Musk – can serve as “important bridges” because they demonstrate the mutual value of collaboration – from technological innovation to job creation – the former chairwoman of the American Chamber of Commerce in China has suggested. Advertisement Roberta Lipson, whose term as chair of AmCham China concluded at the end of the 2024, shortly after this interview, said “trust and understanding” fostered through connections like Musk’s “remain vital” for the broader bilateral relationship in a time of geopolitical challenges. “It is crucial for both governments to maintain open channels of dialogue and focus on creating a predictable business environment,” Lipson said, going on to describe tariffs and broader trade policies as “headwinds” for American business over the past few years. “We encourage the [incoming Trump] administration to prioritise stability in the bilateral relationship,” she said in a written reply to questions from the Post ahead of Trump’s inauguration on January 20. “By focusing on pragmatic economic policies and facilitating constructive dialogue, both sides can ensure a more positive environment for business.” US president-elect Donald Trump threatened to slap 60 per cent tariffs on all imports from China during his campaign, and in a social media post after winning November’s presidential election he threatened to hit Chinese goods with an extra 10 per cent tariff on top of that. Advertisement Lipson said high tariffs “are not a sustainable solution” when dealing with China’s overcapacity issue due to the “unintended consequences for businesses and consumers alike”, and that multilateral engagement and the World Trade Organization offered effective paths to resolution.")
print(jtxt)         # stringa JSON
print(jobj)         # dict (se parsing ok)
#

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rispondi SOLO con JSON valido.

<ARTICLE>American business leaders with “significant” investments in both the US and China – such as Elon Musk – can serve as “important bridges” because they demonstrate the mutual value of collaboration – from technological innovation to job creation – the former chairwoman of the American Chamber of Commerce in China has suggested. Advertisement Roberta Lipson, whose term as chair of AmCham China concluded at the end of the 2024, shortly after this interview, said “trust and understanding” fostered through connections like Musk’s “remain vital” for the broader bilateral relationship in a time of geopolitical challenges. “It is crucial for both governments to maintain open channels of dialogue and focus on creating a predictable business environment,” Lipson said, going on to describe tariffs and broader trade policies as “headwinds” for American business over the past few years. “We encourage the [incoming Trump] administration to prioritise stability

In [ ]:
# Expected: "output": {"overall_sentiment_score": 0.378395, "overall_sentiment_label": "Bullish", "tickers": [{"ticker": "TSLA", "relevance_score": 0.042346, "ticker_sentiment_score": 0.080928, "ticker_sentiment_label": "Neutral"}]}}


In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, json

base_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapters = "out-mistral-json/final"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(adapters); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, adapters).eval()

def infer(article:str):
    msgs=[{"role":"system","content":"Rispondi SOLO con JSON valido."},
          {"role":"user","content": f"<ARTICLE>{article}</ARTICLE>"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    with torch.inference_mode():
        out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                             max_new_tokens=200, temperature=0,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    return txt, (json.loads(txt) if txt.startswith("{") else None)

# Esempio:
jtxt, jobj = infer("New vehicle models and vehicle refreshes strengthen the Tesla bull case for one analyst. - FSD licensing and revenue opportunities add to the long-term outlook. - See what Wall Street is buying with instant access to ratings on 1,000 top stocks, including Goldman Sachs, Morgan Stanley, and more. Unlock all ratings now. A Tesla Inc TSLA analyst sees upcoming revamped and new vehicle models helping to boost sales along with Full Self-Driving licensing potential, brushing aside the company's recent fourth-quarter delivery miss. The Tesla Analyst: Tesla analyst Stephen Gengaro reiterated a Buy rating on Tesla. He also raised the price target from $441 to $492. The Analyst Takeaways: The upcoming rollout of the \"Model 2\" from Tesla is a catalyst, Gengaro said in a new investor note. Strong demand is anticipated for the lower-priced Model 2, according to the analyst. The introduction of this new vehicle could serve as a new way for Tesla vehicles to be more accessible to a wider range of consumers. The analyst also highlighted the company's revamped Model 3 and a Model Y refresh as items that could boost vehicle sales, as reported by Investing.com. \"We believe Tesla is very well positioned to deliver robust multi-year growth in 2025-27+,\" Gengaro said in the note, as shared by Sawyer Merritt. The analyst considers the potential removal of $7,500 in electric vehicle tax credits under the new White House administration as an advantage for Tesla over competitors. Outside of vehicle sales, the analyst also highlights Tesla's Full Self-Driving and artificial intelligence initiatives that could bring value and revenue to the company and its shares. \"We also believe Tesla's AI-based Full Self-Driving initiative has the potential to generate significant value through both sales of FSD, possible licensing agreements, and a critical part of longer-term Cybercab initiatives.\" The analyst's price target increase comes after the delivery miss and the company posting its first annual sales decline on a year-over-year basis. Analysts continue to debate what the short-term and long-term looks like for the electric vehicle leader with new vehicles, robotaxis and FSD among the most covered topics. Tesla will report fourth-quarter financial results after the market closes on Jan. 29. Price Action: Tesla stock was up 0.2% to $411.05 on Monday versus a 52-week trading range of $138.80 to $488.54. Tesla stock is up 71% over the last year. Read Next: Image via Tesla Edge Rankings Price Trend © 2025 Benzinga.com. Benzinga does not provide investment advice. All rights reserved.")
print(jtxt)         # stringa JSON
print(jobj)         # dict (se parsing ok)
#"output": {"overall_sentiment_score": 0.272467, "overall_sentiment_label": "Somewhat-Bullish", "tickers": [{"ticker": "TSLA", "relevance_score": 0.846873, "ticker_sentiment_score": 0.42421, "ticker_sentiment_label": "Bullish"}]}}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rispondi SOLO con JSON valido.

<ARTICLE>New vehicle models and vehicle refreshes strengthen the Tesla bull case for one analyst. - FSD licensing and revenue opportunities add to the long-term outlook. - See what Wall Street is buying with instant access to ratings on 1,000 top stocks, including Goldman Sachs, Morgan Stanley, and more. Unlock all ratings now. A Tesla Inc TSLA analyst sees upcoming revamped and new vehicle models helping to boost sales along with Full Self-Driving licensing potential, brushing aside the company's recent fourth-quarter delivery miss. The Tesla Analyst: Tesla analyst Stephen Gengaro reiterated a Buy rating on Tesla. He also raised the price target from $441 to $492. The Analyst Takeaways: The upcoming rollout of the "Model 2" from Tesla is a catalyst, Gengaro said in a new investor note. Strong demand is anticipated for the lower-priced Model 2, according to the analyst. The introduction of this new vehicle could serve as a new way for Tesla vehicles to b

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, json

base_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapters = "out-mistral-json/final"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(adapters); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, adapters).eval()

def infer(article:str):
    msgs=[{"role":"system","content":"Rispondi SOLO con JSON valido."},
          {"role":"user","content": f"<ARTICLE>{article}</ARTICLE>"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    with torch.inference_mode():
        out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                             max_new_tokens=200, temperature=0,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    return txt, (json.loads(txt) if txt.startswith("{") else None)

# Esempio:
jtxt, jobj = infer("The Invesco S&P 100 Equal Weight ETF (EQWL - Free Report) was launched on 12/01/2006, and is a smart beta exchange traded fund designed to offer broad exposure to the Style Box - Large Cap Blend category of the market. What Are Smart Beta ETFs? The ETF industry has traditionally been dominated by products based on market capitalization weighted indexes that are designed to represent the market or a particular segment of the market. Market cap weighted indexes offer a low-cost, convenient, and transparent way of replicating market returns, and are a good option for investors who believe in market efficiency. However, some investors believe in the possibility of beating the market through exceptional stock selection, and choose a different type of fund that tracks non-cap weighted strategies: smart beta. Non-cap weighted indexes try to choose stocks that have a better chance of risk-return performance, which is based on specific fundamental characteristics, or a mix of other such characteristics. Methodologies like equal-weighting, one of the simplest options out there, fundamental weighting, and volatility/momentum based weighting are all choices offered to investors in this space, but not all of them can deliver superior returns. Fund Sponsor & Index Because the fund has amassed over $951.80 million, this makes it one of the larger ETFs in the Style Box - Large Cap Blend. EQWL is managed by Invesco. Before fees and expenses, this particular fund seeks to match the performance of the Russell Top 200 Equal Weight Index. The S&P 100 Equal Weight Index is designed to provide equal-weighted exposure to the securities of the largest 200 companies in the US equity market. Cost & Other Expenses Cost is an important factor in selecting the right ETF, and cheaper funds can significantly outperform their more expensive cousins if all other fundamentals are the same. Operating expenses on an annual basis are 0.25% for EQWL, making it on par with most peer products in the space. It has a 12-month trailing dividend yield of 1.86%. Sector Exposure and Top Holdings Most ETFs are very transparent products, and disclose their holdings on a daily basis. ETFs also offer diversified exposure, which minimizes single stock risk, though it's still important for investors to research a fund's holdings. This ETF has heaviest allocation in the Financials sector - about 20.40% of the portfolio. Information Technology and Industrials round out the top three. Looking at individual holdings, Tesla Inc (TSLA - Free Report) accounts for about 1.44% of total assets, followed by Wells Fargo & Co (WFC - Free Report) and Capital One Financial Corp (COF - Free Report) . Its top 10 holdings account for approximately 12.61% of EQWL's total assets under management. Performance and Risk So far this year, EQWL has added about 0.22%, and was up about 19.66% in the last one year (as of 01/07/2025). During this past 52-week period, the fund has traded between $86.61 and $107.72. The fund has a beta of 0.97 and standard deviation of 15.40% for the trailing three-year period, which makes EQWL a medium risk choice in this particular space. With about 102 holdings, it effectively diversifies company-specific risk. Alternatives Invesco S&P 100 Equal Weight ETF is a reasonable option for investors seeking to outperform the Style Box - Large Cap Blend segment of the market. However, there are other ETFs in the space which investors could consider. IShares Core S&P 500 ETF (IVV - Free Report) tracks S&P 500 Index and the SPDR S&P 500 ETF (SPY - Free Report) tracks S&P 500 Index. IShares Core S&P 500 ETF has $594.99 billion in assets, SPDR S&P 500 ETF has $635.81 billion. IVV has an expense ratio of 0.03% and SPY charges 0.09%. Investors looking for cheaper and lower-risk options should consider traditional market cap weighted ETFs that aim to match the returns of the Style Box - Large Cap Blend. Bottom Line To learn more about this product and other ETFs, screen for products that match your investment objectives and read articles on latest developments in the ETF investing universe, please visit Zacks ETF Center. See More Zacks Research for These Tickers Normally $25 each - click below to receive one report FREE: Wells Fargo & Company (WFC) - free report >> Capital One Financial Corporation (COF) - free report >> Tesla, Inc. (TSLA) - free report >> SPDR S&P 500 ETF (SPY) - free report >>")
print(jtxt)         # stringa JSON
print(jobj)         # dict (se parsing ok)
#"output": {"overall_sentiment_score": 0.181443, "overall_sentiment_label": "Somewhat-Bullish", "tickers": [{"ticker": "IVZ", "relevance_score": 0.179788, "ticker_sentiment_score": 0.115707, "ticker_sentiment_label": "Neutral"}, {"ticker": "COF", "relevance_score": 0.120431, "ticker_sentiment_score": 0.067868, "ticker_sentiment_label": "Neutral"}, {"ticker": "TSLA", "relevance_score": 0.120431, "ticker_sentiment_score": 0.067868, "ticker_sentiment_label": "Neutral"}, {"ticker": "WFC", "relevance_score": 0.120431, "ticker_sentiment_score": 0.067868, "ticker_sentiment_label": "Neutral"}]}}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rispondi SOLO con JSON valido.

<ARTICLE>The Invesco S&P 100 Equal Weight ETF (EQWL - Free Report) was launched on 12/01/2006, and is a smart beta exchange traded fund designed to offer broad exposure to the Style Box - Large Cap Blend category of the market. What Are Smart Beta ETFs? The ETF industry has traditionally been dominated by products based on market capitalization weighted indexes that are designed to represent the market or a particular segment of the market. Market cap weighted indexes offer a low-cost, convenient, and transparent way of replicating market returns, and are a good option for investors who believe in market efficiency. However, some investors believe in the possibility of beating the market through exceptional stock selection, and choose a different type of fund that tracks non-cap weighted strategies: smart beta. Non-cap weighted indexes try to choose stocks that have a better chance of risk-return performance, which is based on specific fundamental charac

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, json

base_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapters = "out-mistral-json/final"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(adapters); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, adapters).eval()

def infer(article:str):
    msgs=[{"role":"system","content":"Rispondi SOLO con JSON valido."},
          {"role":"user","content": f"<ARTICLE>{article}</ARTICLE>"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    with torch.inference_mode():
        out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                             max_new_tokens=200, temperature=0,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    return txt, (json.loads(txt) if txt.startswith("{") else None)

# Esempio:
jtxt, jobj = infer("Adobe (ADBE - Free Report) is bolstering its music generation and editing capabilities on the back of generative AI. Recently, the company collaborated with researchers at the University of California and Carnegie Mellon to introduce Project Music GenAI Control, a platform that allows users to generate audio from text descriptions or reference melodies. The tool uses generative AI to assist users in creating and editing music without professional experience. Users can input text descriptions, customize music and remix clips, making it ideal for content creators. Further, the music generation tool enables customization of tempo, intensity, repeating patterns and structure. Adobe is expected to gain solid traction across content creators on the back of its latest launch. Stiff Competition The latest move is likely to aid Adobe to strengthen its competitive position against peers like Alphabet (GOOGL - Free Report) , Meta Platforms (META - Free Report) and Microsoft (MSFT - Free Report) , which are also making concerted efforts to leverage generative AI capabilities for music creation. Recently, Alphabet’s Google introduced MusicFX, an enhanced version of its MusicLM tool, aiming to revolutionize the music industry with its superior quality and speed. This generative-AI-powered music generation tool allows users to create 70-second music ditties and loops, with text prompts, suggestions, and a word cloud for instrument and tempo recommendations. Additionally, Alphabet launched TextFX, a lyrics generation tool, in its AI Test Kitchen, collaborating with Lupe Fiasco to enhance the creative process for lyricists and writers. Meanwhile, Meta Platforms is enjoying the growing momentum of its text-to-music generative AI model, MusicGen. MusicGen generates 15 seconds of audio based on user description, producing up to 120 seconds of clips if hosted on HuggingSpace. It can also recreate user-specified melodies by extracting a melody from a reference audio file. Microsoft, on the other hand, partnered with Suno AI to allow its AI chatbot Copilot to create AI songs on demand using the Suno app. Microsoft chatbot users can click on the “Make music with Suno” option to compose original songs based on text prompts using Suno’s Discord tool. Expanding Generative AI-Backed Offering The latest move bodes well for Adobe’s growing efforts to strengthen its generative AI capabilities, which are presently acting as a key growth catalyst for the company. Its shares have gained 68% in the past year compared with the Zacks Computer & Technology sector’s growth of 50.2%. In this regard, Adobe’s launch of the Firefly Image 2 Model, Firefly Vector Model and Firefly Design Model to mark a significant advancement in its creative generative AI model family, enhancing creative control, image quality and illustrator capabilities, remains noteworthy. The company also launched Photoshop’s web version, which is available via Firefly-powered AI tools, generative expand and generative fill. The editing tool enables file collaboration through sharing links, even without subscriptions and provides a web version with desktop tools like the contextual taskbar for workflow suggestions. Moreover, these efforts will likely aid this Zacks Rank #2 (Buy) company in capitalizing on growth opportunities present in the global generative AI market. Per a Grand View Research report, the generative AI market size is expected to witness a CAGR of 36.5% between 2024 and 2030. You can see the complete list of today’s Zacks #1 Rank (Strong Buy) stocks here. Solidifying prospects in the promising generative AI market will, in turn, aid its overall financial performance in the near term. The Zacks Consensus Estimate for fiscal 2024 total revenues stands at $21.41 billion, indicating year-over-year growth of 10.3%. See More Zacks Research for These Tickers Normally $25 each - click below to receive one report FREE: Microsoft Corporation (MSFT) - free report >> Adobe Inc. (ADBE) - free report >>")
print(jtxt)         # stringa JSON
print(jobj)         # dict (se parsing ok)
# "output": {"overall_sentiment_score": 0.365074, "overall_sentiment_label": "Bullish", "tickers": [{"ticker": "ADBE", "relevance_score": 0.348103, "ticker_sentiment_score": 0.348423, "ticker_sentiment_label": "Somewhat-Bullish"}, {"ticker": "MSFT", "relevance_score": 0.236396, "ticker_sentiment_score": 0.214329, "ticker_sentiment_label": "Somewhat-Bullish"}, {"ticker": "META", "relevance_score": 0.178459, "ticker_sentiment_score": 0.255548, "ticker_sentiment_label": "Somewhat-Bullish"}, {"ticker": "GOOG", "relevance_score": 0.059935, "ticker_sentiment_score": 0.205383, "ticker_sentiment_label": "Somewhat-Bullish"}]}}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rispondi SOLO con JSON valido.

<ARTICLE>Adobe (ADBE - Free Report) is bolstering its music generation and editing capabilities on the back of generative AI. Recently, the company collaborated with researchers at the University of California and Carnegie Mellon to introduce Project Music GenAI Control, a platform that allows users to generate audio from text descriptions or reference melodies. The tool uses generative AI to assist users in creating and editing music without professional experience. Users can input text descriptions, customize music and remix clips, making it ideal for content creators. Further, the music generation tool enables customization of tempo, intensity, repeating patterns and structure. Adobe is expected to gain solid traction across content creators on the back of its latest launch. Stiff Competition The latest move is likely to aid Adobe to strengthen its competitive position against peers like Alphabet (GOOGL - Free Report) , Meta Platforms (META - Free Rep